# 02 · Azure OpenAI GPT-5.1 Baseline Evaluation

Run the Text-to-SQL test split through **GPT-5.1** via Azure OpenAI and score
each generated query with the same three-component reward used during GRPO
training (format · execution · schema fidelity).


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

# ── Azure OpenAI ──────────────────────────────────────────────────────────
AZURE_OPENAI_ENDPOINT    = "https://<your-resource>.openai.azure.com/"
AZURE_OPENAI_DEPLOYMENT  = "gpt-5.1"            # Azure deployment name
AZURE_OPENAI_API_VERSION = "2025-01-01-preview"
TEMPERATURE              = 0.0                   # deterministic for reproducibility
MAX_TOKENS               = 512

# ── Data ──────────────────────────────────────────────────────────────────
CSV_PATH = REPO_ROOT / "data" / "test_dataset_baseline.csv"

# Root under which rawdata/spider/... and rawdata/bird/... live.
# Set to "/kaggle/working/" on Kaggle, or your local equivalent.
RAWDATA_DIR = "/kaggle/working/"

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"CSV_PATH   : {CSV_PATH}")


## 1. Setup Azure OpenAI Client


In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

credential     = DefaultAzureCredential(exclude_interactive_browser_credential=False)
token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version=AZURE_OPENAI_API_VERSION,
)

print("Azure OpenAI client ready.")


## 2. Load Test Dataset


In [ ]:
import ast
import pandas as pd

df = pd.read_csv(CSV_PATH)

# The prompt and schema columns are stored as Python repr strings in the CSV.
df["prompt_parsed"] = df["prompt"].apply(ast.literal_eval)
df["schema_parsed"] = df["schema"].apply(ast.literal_eval)

print(f"Loaded {len(df)} rows")
print(f"Sources : {df['source'].value_counts().to_dict()}")
df[["db_id", "source", "solution"]].head(3)


## 3. Run Azure OpenAI Inference


In [ ]:
from tqdm.auto import tqdm


def call_azure_openai(messages: list[dict], temperature: float = 0.0) -> str:
    """Send a chat-format prompt to Azure OpenAI and return the text response."""
    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=messages,
        temperature=temperature,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content or ""


completions = []
for messages in tqdm(df["prompt_parsed"], desc=f"Inference ({AZURE_OPENAI_DEPLOYMENT})"):
    try:
        text = call_azure_openai(messages, TEMPERATURE)
    except Exception as exc:
        print(f"API error: {exc}")
        text = ""
    completions.append(text)

df["gpt_completion"] = completions
print(f"\n{sum(bool(c) for c in completions)}/{len(completions)} non-empty responses.")
df[["db_id", "gpt_completion"]].head(3)


## 4. Compute Rewards


In [ ]:
import rewards as _rewards_mod
from rewards import combined_reward

# Patch _exec_on_sqlite to use RAWDATA_DIR instead of the default "/kaggle/working/".
# This makes the execution reward work both on Kaggle and in a local environment.
_orig_exec = _rewards_mod._exec_on_sqlite
_rewards_mod._exec_on_sqlite = (
    lambda sql, db_path=None, source=None: _orig_exec(sql, db_path, source, base_path=RAWDATA_DIR)
)

# Build TRL-format inputs ─ completions as list[list[dict]]
trl_completions = [[{"role": "assistant", "content": c}] for c in df["gpt_completion"]]
trl_prompts     = df["prompt_parsed"].tolist()
schemas         = df["schema_parsed"].tolist()
sources         = df["source"].tolist()
db_ids          = df["db_id"].tolist()

scores = combined_reward(
    completions=trl_completions,
    prompts=trl_prompts,
    schemas=schemas,
    db_paths=db_ids,
    source=sources,
    weights={"format": 0.2, "exec": 0.5, "schema_fidelity": 0.3},
)

df["gpt_reward"] = scores
print(df["gpt_reward"].describe().to_string())


## 5. Average Reward


In [ ]:
overall_avg = df["gpt_reward"].mean()
per_source  = df.groupby("source")["gpt_reward"].mean()

print(f"── GPT-5.1 Reward Summary ──────────────────────────────")
print(f"  Overall average : {overall_avg:.4f}")
print()
print("  Per-source average:")
for src, val in per_source.items():
    print(f"    {src:10s}: {val:.4f}")

# Side-by-side comparison if the CSV already contains pre-computed rewards.
if "reward" in df.columns:
    print()
    print("  Comparison (pre-computed in CSV vs GPT-5.1):")
    comparison = df.groupby("source")[["reward", "gpt_reward"]].mean()
    comparison.columns = ["pre_computed", "gpt_5_1"]
    print(comparison.round(4).to_string())
